In [1]:
# importing libraries
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
import numpy as np
from sklearn.metrics import classification_report
from datasets import load_from_disk
import evaluate


In [2]:
# Loading dataset
dataset_dict = load_from_disk("E:/PROJECTS/Privacy-Risk-Analyser/privacy_ner_dataset")

In [3]:
# Loading tokenizer and define label list
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = dataset_dict["train"].features["labels"].feature.names
label_to_id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)

In [4]:
# Tokenize with aligned labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


# Mapping dataset
tokenized_datasets = dataset_dict.map(tokenize_and_align_labels, batched=True)

In [5]:
# Loading model
from transformers import XLMRobertaForTokenClassification

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = XLMRobertaForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from collections import Counter
label_counts = Counter([l for label_seq in tokenized_datasets["train"]["labels"] for l in label_seq if l != -100])
print({id2label[k]: v for k, v in label_counts.items()})

{'B-NAME': 2222, 'I-NAME': 2839, 'O': 309628, 'B-PHONE': 243, 'I-PHONE': 1032, 'B-EMAIL': 494, 'I-EMAIL': 4310, 'B-ADDRESS': 173, 'I-ADDRESS': 1018}


In [7]:

data_collator = DataCollatorForTokenClassification(tokenizer)

# Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }



In [8]:
'''from sklearn.utils.class_weight import compute_class_weight
import torch

# Flatten all label ids across training set, skipping -100
all_labels = [
    label
    for example in tokenized_datasets["train"]["labels"]
    for label in example if label != -100
]

# Compute weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
'''

'from sklearn.utils.class_weight import compute_class_weight\nimport torch\n\n# Flatten all label ids across training set, skipping -100\nall_labels = [\n    label\n    for example in tokenized_datasets["train"]["labels"]\n    for label in example if label != -100\n]\n\n# Compute weights\nclass_weights = compute_class_weight(\n    class_weight=\'balanced\',\n    classes=np.unique(all_labels),\n    y=all_labels\n)\nclass_weights = torch.tensor(class_weights, dtype=torch.float)\n'

In [9]:
from collections import Counter
import torch
import numpy as np

# Step 1: Count label occurrences, ignoring special tokens
label_counts = Counter(
    l for example in tokenized_datasets["train"]["labels"]
    for l in example if l != -100
)

# Step 2: Label mappings
label_list = dataset_dict["train"].features["labels"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Step 3: Compute class weights
total = sum(label_counts.values())
class_weights = []

boost_factor = 2.0  # Boost B- labels

for i, label in enumerate(label_list):
    count = label_counts.get(i, 1)  # Avoid divide-by-zero
    weight = total / count

    # Boost B-labels
    if label.startswith("B-"):
        weight *= boost_factor

    # Optional: Log-scaling for smoother distribution
    weight = np.log1p(weight)  # log(1 + weight)

    class_weights.append(weight)

# Step 4: Convert to tensor and normalize
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = class_weights / class_weights.sum()

# Optional: Print for verification
for i, w in enumerate(class_weights):
    print(f"{id2label[i]:>10}: {w:.4f}")


 B-ADDRESS: 0.1637
   B-EMAIL: 0.1428
    B-NAME: 0.1129
   B-PHONE: 0.1569
 I-ADDRESS: 0.1147
   I-EMAIL: 0.0861
    I-NAME: 0.0943
   I-PHONE: 0.1144
         O: 0.0142


In [10]:
from transformers import XLMRobertaForTokenClassification
import torch.nn as nn

model = XLMRobertaForTokenClassification.from_pretrained("xlm-roberta-base", num_labels=len(label_list))

# loss function
def custom_compute_loss(model, inputs, return_outputs=False):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits

    loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device), ignore_index=-100)
    loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

    return (loss, outputs) if return_outputs else loss


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
from transformers import Trainer
import torch.nn as nn

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Use the correct weight tensor (assumes you named it `class_weights`)
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


In [12]:
from transformers import TrainingArguments

train_dataset = tokenized_datasets["train"].select(range(200))
eval_dataset = tokenized_datasets["validation"].select(range(50))

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=False,  

    num_train_epochs=2,
    per_device_train_batch_size=4,     
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=100
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


C:\Users\Dell\AppData\Local\Temp\ipykernel_9708\3974548928.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


Step,Training Loss
100,0.426100
200,0.135300
300,0.093000
400,0.081200


TrainOutput(global_step=400, training_loss=0.1839078903198242, metrics={'train_runtime': 6449.2303, 'train_samples_per_second': 0.248, 'train_steps_per_second': 0.062, 'total_flos': 415755574189704.0, 'train_loss': 0.1839078903198242, 'epoch': 2.0})

In [13]:
trainer.save_model("E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner")
tokenizer.save_pretrained("E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner")

('E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\tokenizer_config.json',
 'E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\special_tokens_map.json',
 'E:/PROJECTS/Privacy-Risk-Analyser/models/xlm-roberta-ner\\tokenizer.json')